In [1]:
!git clone https://github.com/yumelab-studio/avaia-seismofinance.git

Cloning into 'avaia-seismofinance'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 130 (delta 42), reused 4 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 1.69 MiB | 7.81 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [2]:
import os
os.chdir('avaia-seismofinance')

In [3]:
import pandas as pd

df_eq=pd.read_csv('data/raw/earthquake.csv')
df_eq.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-04-04T10:36:46.191Z,38.4373,141.9269,67.953,5.0,mb,43.0,94.0,2.442,1.03,...,2026-04-04T10:58:55.040Z,"41 km E of Onagawa Chō, Japan",earthquake,7.46,7.045,0.031,334.0,reviewed,us,us
1,2026-04-02T00:52:52.373Z,38.1670,141.6037,66.591,5.0,mb,58.0,128.0,2.652,0.55,...,2026-04-03T00:41:21.454Z,"33 km SSE of Onagawa Chō, Japan",earthquake,16.88,21.079,0.060,88.0,reviewed,us,us
2,2026-03-27T15:24:33.227Z,39.3975,143.1958,16.096,5.1,mb,55.0,135.0,2.146,1.01,...,2026-03-27T15:39:29.040Z,"107 km E of Yamada, Japan",earthquake,7.86,5.085,0.062,83.0,reviewed,us,us
3,2026-03-26T14:18:50.947Z,39.4462,143.3717,9.513,6.5,mww,121.0,45.0,2.220,0.82,...,2026-03-27T14:47:28.298Z,"122 km E of Yamada, Japan",earthquake,6.84,4.170,0.038,66.0,reviewed,us,us
4,2026-03-21T11:12:04.314Z,29.4858,130.6689,21.650,5.4,mww,98.0,110.0,2.025,0.87,...,2026-04-02T11:41:55.694Z,"84 km S of Koshima, Japan",earthquake,6.94,4.967,0.063,24.0,reviewed,us,us


In [4]:
df_eq = df_eq[['time', 'latitude', 'longitude', 'depth', 'mag', 'place', 'type', 'status']]

In [5]:
df_eq = df_eq.rename(columns={
    'time': 'datetime_utc',
    'depth': 'depth_km',
    'mag': 'magnitude'
})

In [6]:
df_eq['datetime_utc'] = pd.to_datetime(df_eq['datetime_utc'], utc=True)
df_eq['datetime_jst'] = df_eq['datetime_utc'].dt.tz_convert('Asia/Tokyo')
df_eq['date_jst'] = df_eq['datetime_jst'].dt.date

In [7]:
df_eq = df_eq.dropna(subset=['datetime_jst', 'latitude', 'longitude', 'depth_km', 'magnitude'])
df_eq = df_eq.drop_duplicates()

In [8]:
df_eq = df_eq[df_eq['magnitude'] >= 5]

In [10]:
df_eq = df_eq.sort_values(by='datetime_utc')

In [11]:
df_eq.to_csv('data/clean/earthquake_clean_v2.csv', index=False)

In [14]:
#check
df_check = pd.read_csv('data/clean/earthquake_clean_v2.csv')
df_check.head()

,datetime_utc,latitude,longitude,depth_km,magnitude,place,type,status,datetime_jst,date_jst
0,2011-04-06 13:54:52.200000+00:00,37.654,141.425,59.6,5.3,"42 km ENE of Namie, Japan",earthquake,reviewed,2011-04-06 22:54:52.200000+09:00,2011-04-06
1,2011-04-07 14:32:43.290000+00:00,38.276,141.588,42.0,7.1,"29 km ESE of Ishinomaki, Japan",earthquake,reviewed,2011-04-07 23:32:43.290000+09:00,2011-04-07
2,2011-04-07 22:02:19.880000+00:00,39.142,142.893,40.9,5.1,"88 km ESE of Yamada, Japan",earthquake,reviewed,2011-04-08 07:02:19.880000+09:00,2011-04-08
3,2011-04-09 01:25:26.660000+00:00,36.750,142.049,18.7,5.1,"109 km ESE of Iwaki, Japan",earthquake,reviewed,2011-04-09 10:25:26.660000+09:00,2011-04-09
4,2011-04-09 09:42:21.890000+00:00,38.280,141.701,65.3,5.3,"38 km ESE of Ishinomaki, Japan",earthquake,reviewed,2011-04-09 18:42:21.890000+09:00,2011-04-09


In [15]:
stock = pd.read_csv('data/raw/stocks.csv', header=[0, 1], index_col=0)
stock.columns = ['_'.join(col).strip() for col in stock.columns]
stock = stock.reset_index()
stock.columns = [c.lower() for c in stock.columns]

In [16]:
stock = stock.rename(columns={
    'price': 'date',
    'close_8766.t': 'close',
    'volume_8766.t': 'volume'
})
stock = stock[['date', 'close', 'volume']]

In [17]:
stock['date'] = pd.to_datetime(stock['date'], errors='coerce')
stock = stock.dropna(subset=['date'])

In [18]:
stock['ticker'] = '8766.T'
stock = stock.sort_values('date')
stock['return'] = stock['close'].pct_change()
stock = stock.dropna(subset=['return'])

In [19]:
stock.to_csv('data/clean/stocks_clean_v2.csv', index=False)

In [ ]:
## Market Index Data — Nikkei 225 (^N225)

In [20]:
import yfinance as yf
market = yf.download("^N225", start="2010-01-01")
market.to_csv('data/raw/market_index.csv')

/tmp/ipykernel_10375/879543718.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  market = yf.download("^N225", start="2010-01-01")
[*********************100%***********************]  1 of 1 completed


In [22]:

# Download directly
market_raw = yf.download("^N225", start="2010-01-01")
market_raw.to_csv('data/raw/market_index.csv')

# Work with the live DataFrame — no header parsing needed
market = market_raw[['Close']].copy()
market.columns = ['market_close']
market.index.name = 'date'
market = market.reset_index()
market['date'] = pd.to_datetime(market['date'])
market = market.sort_values('date')
market['market_return'] = market['market_close'].pct_change()
market = market.dropna(subset=['market_return'])
market.to_csv('data/clean/market_index_clean.csv', index=False)
print(market.head())

/tmp/ipykernel_10375/418823677.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  market_raw = yf.download("^N225", start="2010-01-01")
[*********************100%***********************]  1 of 1 completed


        date  market_close  market_return
1 2010-01-05  10681.830078       0.002538
2 2010-01-06  10731.450195       0.004645
3 2010-01-07  10681.660156      -0.004640
4 2010-01-08  10798.320312       0.010922
5 2010-01-12  10879.139648       0.007484


In [23]:
# Quality checks
print("=== Earthquake ===")
eq_v2 = pd.read_csv('data/clean/earthquake_clean_v2.csv')
print(f"Rows: {len(eq_v2)}")
print(f"Columns: {list(eq_v2.columns)}")
print(f"Missing values:\n{eq_v2.isnull().sum()}")
print(f"Date range: {eq_v2['date_jst'].min()} to {eq_v2['date_jst'].max()}")

print("\n=== Stocks ===")
stk_v2 = pd.read_csv('data/clean/stocks_clean_v2.csv')
print(f"Rows: {len(stk_v2)}")
print(f"Missing values:\n{stk_v2.isnull().sum()}")
print(f"Date range: {stk_v2['date'].min()} to {stk_v2['date'].max()}")

print("\n=== Market Index ===")
mkt = pd.read_csv('data/clean/market_index_clean.csv')
print(f"Rows: {len(mkt)}")
print(f"Date range: {mkt['date'].min()} to {mkt['date'].max()}")

=== Earthquake ===
Rows: 1330
Columns: ['datetime_utc', 'latitude', 'longitude', 'depth_km', 'magnitude', 'place', 'type', 'status', 'datetime_jst', 'date_jst']
Missing values:
datetime_utc    0
latitude        0
longitude       0
depth_km        0
magnitude       0
place           0
type            0
status          0
datetime_jst    0
date_jst        0
dtype: int64
Date range: 2011-04-06 to 2026-04-04

=== Stocks ===
Rows: 3999
Missing values:
date      0
close     0
volume    0
ticker    0
return    0
dtype: int64
Date range: 2010-01-05 to 2026-04-10

=== Market Index ===
Rows: 4005
Date range: 2010-01-05 to 2026-05-25
